# 05 · Brass experiment: nine nominal runs, resumed automatically

**Run All is configured for the compact production study.** This notebook completes **three methods × three training seeds = nine nominal trainings**. It checks saved model identities first, reuses completed jobs, and resumes compatible interrupted jobs. The limit counts pending jobs after completed jobs are removed. The expensive optional sensitivity and classical studies stay off.

The temperatures are digitized profiles from **one experimental run**. Training uses the **48 interior terminal temperatures** and the **two measured endpoint histories**. The 48 initial and 192 intermediate interior values are reserved for assessment; endpoints already prescribe boundary data and are excluded from assessment.

Every completed nominal candidate is frozen before withheld assessment. All nine candidates receive physical-unit evaluation, independent forward replay and spatial/time refinement; these calculations are cached as well. Three training repeats give descriptive evidence on this dataset, not experimental replication or precise confidence intervals. No result is selected because its initial error is low.

Keep the production settings below unchanged to retain compatibility with saved runs. The original calibration and digitization questions remain author tasks.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in _candidates if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted heat folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import write_json, read_json, array_hash, environment
from inverse_study.brass import (load_brass, baseline_observations, physical_metrics,
    evaluate_saved_pinn, replay_initial, replay_refinement, sensitivity_scenarios,
    retraining_plan, freeze_candidate, SCALE_K, OFFSET_K)
from inverse_study.pinn import train_inverse, predict_saved, tensorflow_available
from inverse_study import brass_completion as brass_done
from inverse_study.numerics import nonlinear_tikhonov
from inverse_study.qr_audit import solve_qr_1d, select_qr_by_discrepancy

DATA = ROOT / "data"
RESULTS = ROOT / "results"
EXPORTS = RESULTS / "brass" / "source_audit"
EXPORTS.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("TensorFlow available:", tensorflow_available())
print("No package installation is performed by this notebook.")

## 1. Execution controls

The compact study uses **three hidden layers of width 60, 16 Fourier frequencies per coordinate, 60,000 Adam steps, 12,000 residual points and 200 boundary points**, exactly as in the previous production configuration. Training is enabled below. Set `RUN_TRAINING=False` for an audit-only pass; completed evaluations/replays still reuse their caches.

`MAX_NEW_TRAINING_JOBS=None` completes all pending nominal jobs. Set it to `1` to run or resume **one pending job** per execution; already completed jobs do not consume this allowance. The plan remains nine runs either way. A completed training is a candidate, not automatically a publication result.

`UNCERTAINTY_K=None` remains intentional. A classical discrepancy fit requires documented observational uncertainty and an explicitly reviewed per-candidate numerical-floor record. No synthetic experimental noise is invented.

In [ ]:
PROFILE = "production"
ALLOW_PRODUCTION = True
RUN_TRAINING = True
MAX_NEW_TRAINING_JOBS = None       # Count only missing/resumable jobs; 1 advances one pending job.
METHODS = ("plain", "fourier_fixed", "fourier_adaptive")
TRAINING_SEEDS = (2026, 2027, 2028)

RUN_EVALUATION = True
EVALUATION_RUN_DIRS = []           # Empty: every completed candidate in this exact nine-run plan.
RUN_REPLAY = True
RUN_REFINEMENT = True
REPLAY_MESHES = (101, 201, 401)
REPLAY_RTOL, REPLAY_ATOL, REPLAY_MAX_STEP = 1e-8, 1e-10, 0.02
NUMERICAL_FLOOR_FACTOR = 2.0       # Observed sensitivity guard, not a certified error bound.
ACCEPT_OBSERVED_NUMERICAL_FLOOR = False

UNCERTAINTY_K = None
UNCERTAINTY_INTERPRETATION = "pointwise_bound"
UNCERTAINTY_PROVENANCE = ""
RUN_NONLINEAR_TIKHONOV = False
RUN_QR_AUDIT = False

RUN_SENSITIVITY = False
MAX_NEW_SENSITIVITY_JOBS = 1       # Optional; also counts pending jobs only.
SENSITIVITY_METHODS = ("fourier_adaptive",)
SENSITIVITY_SEEDS = (2026,)
SENSITIVITY_NAMES = []

if PROFILE not in {"smoke", "production"}:
    raise ValueError("PROFILE must be smoke or production")
if PROFILE == "production" and not ALLOW_PRODUCTION and any((RUN_TRAINING, RUN_SENSITIVITY, RUN_NONLINEAR_TIKHONOV, RUN_QR_AUDIT)):
    raise ValueError("Production work is disabled by ALLOW_PRODUCTION.")
BRASS_RESULTS = RESULTS / "brass" / PROFILE
BRASS_RESULTS.mkdir(parents=True, exist_ok=True)

## 2. Load and verify the observational split

The nondimensional variables are
\[
\xi=x/(0.034\,\mathrm{m}),\qquad \tau=t/(9\,\mathrm{s}),\qquad
\theta=(T_K-520\,\mathrm{K})/(190\,\mathrm{K}).
\]
The problem is \(\theta_\tau=\partial_\xi[(B_0+B_1\theta)\theta_\xi]\), with \(B_0=0.287265051903114\), \(B_1=0.011656401384083\), and measured Dirichlet histories. The loader checks these coefficients against the dimensional law in the supplied CSV and checks every profile endpoint against the boundary-history file.

**Calibration remains unresolved.** The supplied coefficient file says the relation was selected using the 3.6–5.4 s inversion interval. That interval overlaps the same run used here for withheld-profile assessment. Verify the original article's calibration inputs, including whether the initial profile was used. This notebook does not claim independent coefficient calibration.

In [ ]:
problem, observations, heldout, provenance = load_brass(DATA / "brass")
OBSERVATION_HASH = array_hash(observations["X_obs"], observations["y_obs"])
assert observations["X_obs"].shape == (48, 1)
assert observations["y_obs"].shape == (48,)
assert len(heldout) == 240 and heldout.groupby("time_s").size().eq(48).all()
assert observations["noise_norm"] is None
split = pd.DataFrame([
    {"role": "terminal interior training", "points": 48, "time_s": "9"},
    {"role": "known endpoint histories", "points": 12, "time_s": "0,1.8,3.6,5.4,7.2,9"},
    {"role": "withheld initial interiors", "points": 48, "time_s": "0"},
    {"role": "withheld same-run intermediate interiors", "points": 192, "time_s": "1.8,3.6,5.4,7.2"},
])
display(split)
print("Declared coefficient source:", provenance["coefficient_source_note"])
print("Calibration independence:", provenance["calibration_independence"])
print("Noise/digitization uncertainty:", provenance["uncertainty_status"])
print("Observation identity:", OBSERVATION_HASH)
write_json(EXPORTS / "brass_source_provenance.json", provenance)
write_json(EXPORTS / "execution_environment.json", environment())

In [ ]:
# Plot ONLY information available to the inverse objective before candidate freezing.
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7), constrained_layout=True)
x_mm = observations["X_obs"][:, 0] * 34.0
ax[0].plot(x_mm, OFFSET_K + SCALE_K * observations["y_obs"], "o", ms=3, label="Interior terminal observations")
ax[0].set(xlabel="Axial position (mm)", ylabel="Temperature (K)", title="Training terminal profile: 9 s")
ax[0].legend(fontsize=8)
times_s = np.asarray(problem["boundary_times"]) * 9.0
ax[1].plot(times_s, OFFSET_K + SCALE_K*np.asarray(problem["boundary_left"]), "o-", label="Left endpoint")
ax[1].plot(times_s, OFFSET_K + SCALE_K*np.asarray(problem["boundary_right"]), "s--", label="Right endpoint")
ax[1].set(xlabel="Time (s)", ylabel="Temperature (K)", title="Prescribed boundary histories")
ax[1].legend(fontsize=8)
for axis in ax: axis.grid(alpha=.25)
fig.savefig(EXPORTS / "brass_training_data_K.png", dpi=180)
plt.show()

### Export the supplied data and provenance checklist

`data.csv` is a reorganized copy of the six supplied profiles, with explicit training/withheld roles. It contains **no predictions**. The endpoint histories appear once per endpoint and time. The checklist records unresolved source issues; merely running this notebook does not resolve them.

In [ ]:
terminal_rows = pd.DataFrame(dict(x_m=observations["X_obs"][:,0]*0.034,
    time_s=9.0, temperature_K=OFFSET_K+SCALE_K*observations["y_obs"], role="training_terminal_interior"))
boundary_rows = pd.DataFrame([
    dict(x_m=x, time_s=t*9.0, temperature_K=OFFSET_K+SCALE_K*v, role="training_boundary_history")
    for x, values in [(0.0, problem["boundary_left"]), (0.034, problem["boundary_right"])]
    for t, v in zip(problem["boundary_times"], values)
])
source_data = pd.concat([heldout[["x_m", "time_s", "temperature_K", "role"]], terminal_rows, boundary_rows], ignore_index=True)
source_data = source_data.sort_values(["time_s", "x_m"]).reset_index(drop=True)
assert len(source_data) == 300 and not source_data.duplicated(["time_s", "x_m"]).any()
source_data.to_csv(EXPORTS / "data.csv", index=False)
checklist_path = EXPORTS / "source_checklist.csv"
if checklist_path.exists():
    checklist = pd.read_csv(checklist_path).fillna("")
    if not {"check", "status", "author_evidence"}.issubset(checklist.columns):
        raise ValueError("Existing source_checklist.csv has an unexpected schema; preserve and review it manually.")
else:
    checklist = pd.DataFrame({"check": provenance["pending_checks"], "status": "unresolved", "author_evidence": ""})
    checklist.to_csv(checklist_path, index=False)
split.to_csv(EXPORTS / "observation_split.csv", index=False)
display(checklist)
print("Source exports:", EXPORTS)

## 3. Complete only missing nominal neural runs

All three methods receive the same terminal observation vector and measured boundary histories. Seeds 2026, 2027 and 2028 are fixed prospectively; no seed is retained or discarded based on withheld errors. The queue verifies source/configuration/data signatures and completed artifacts. A conflict is reported for review instead of silently creating an alternative training run.

Initial/intermediate interior measurements never enter the trainer. These repetitions measure training variability on one experimental dataset. The existing fixed-budget final checkpoint remains the primary endpoint.

In [ ]:
NOMINAL_PLAN = retraining_plan(problem, training_seeds=TRAINING_SEEDS, methods=METHODS,
                               include_sensitivity=False, profile=PROFILE)
plan_table = pd.DataFrame([dict(run_id=j["config"]["run_id"], method=j["config"]["method"],
    training_seed=j["config"]["training_seed"], profile=j["config"]["profile"],
    epochs=j["config"]["epochs"], N_r=j["config"]["N_r"], N_b=j["config"]["N_b"],
    batch_r=j["config"]["batch_r"], batch_b=j["config"]["batch_b"],
    hidden_depth=j["config"]["hidden_depth"], hidden_width=j["config"]["hidden_width"])
    for j in NOMINAL_PLAN])
nominal_inventory = brass_done.job_inventory(NOMINAL_PLAN, observations, BRASS_RESULTS)
display(plan_table)
display(nominal_inventory[["method", "training_seed", "status", "reason"]])
plan_table.to_csv(BRASS_RESULTS / "brass_nominal_plan.csv", index=False)
nominal_inventory.to_csv(BRASS_RESULTS / "brass_nominal_queue.csv", index=False)
print("Planned nominal jobs:", len(NOMINAL_PLAN), "| Pending-job limit:", MAX_NEW_TRAINING_JOBS)

In [ ]:
if RUN_TRAINING:
    nominal_inventory, training_outcomes = brass_done.train_missing(
        NOMINAL_PLAN, observations, BRASS_RESULTS, max_new_jobs=MAX_NEW_TRAINING_JOBS)
    display(training_outcomes if len(training_outcomes) else pd.DataFrame({"status": ["No training needed."]}))
else:
    nominal_inventory = brass_done.job_inventory(NOMINAL_PLAN, observations, BRASS_RESULTS)
    print("Training disabled; saved results are still available for assessment.")
nominal_inventory.to_csv(BRASS_RESULTS / "brass_nominal_queue.csv", index=False)
completed_nominal_runs = [Path(p) for p in nominal_inventory.loc[
    nominal_inventory.status.eq("completed"), "run_dir"]]
print("Completed nominal candidates:", len(completed_nominal_runs), "/", len(NOMINAL_PLAN))

## 4. Freeze all available candidates, then assess withheld interiors

The final fixed-budget checkpoint identities are frozen before this assessment loop starts. The freeze stores hashes and a declared selection rule. It does not prove that a person has never inspected heldout values. Existing candidate identities remain protected, and moving the extracted folder between machines does not invalidate an otherwise identical candidate.

Errors at **0, 1.8, 3.6, 5.4 and 7.2 s** are reported in kelvin: RMSE, MAE, signed bias and maximum absolute error. The historical shifted relative norm is retained under its explicit name. Cached results are reused only when weights, data and assessment configuration match.

An empty `EVALUATION_RUN_DIRS` selects all completed candidates in the declared nine-run plan, including previously completed ones. An explicit list may select a subset of those candidates for inspection; it does not change the study plan.

In [ ]:
def _project_path(value):
    p = Path(value)
    return p.resolve() if p.is_absolute() else (ROOT / p).resolve()

evaluation_runs = []
analysis_failures = []
if RUN_EVALUATION:
    allowed = {p.resolve() for p in completed_nominal_runs}
    evaluation_runs = [_project_path(p) for p in EVALUATION_RUN_DIRS] if EVALUATION_RUN_DIRS else list(completed_nominal_runs)
    if any(p.resolve() not in allowed for p in evaluation_runs):
        raise ValueError("Evaluation selection must belong to the completed compact nominal plan.")
    for run_dir in evaluation_runs:
        brass_done.record_training_identity(run_dir, observations, scenario="nominal")
        brass_done.freeze_candidate_portable(run_dir)
    print("Candidates frozen before evaluation:", len(evaluation_runs))
    if not evaluation_runs:
        print("No completed nominal candidates yet; the declared missing jobs remain in the queue.")
else:
    print("Withheld evaluation disabled.")

In [ ]:
assessed_runs = []
if RUN_EVALUATION:
    for run_dir in evaluation_runs:
        try:
            metrics, points, cache_status = brass_done.assessment_cached(run_dir, observations, heldout)
        except Exception as exc:
            analysis_failures.append({"run": run_dir.name, "step": "withheld_assessment", "error": str(exc)})
            print("Assessment failed:", run_dir.name, str(exc))
            continue
        assessed_runs.append(run_dir)
        print(run_dir.name, "assessment:", cache_status)
        display(metrics)
        if cache_status != "cached" or not (run_dir / "brass_assessment_K.png").exists():
            fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
            g = points[np.isclose(points.time_s, 0)]
            ax[0].plot(g.x_m*1000, g.temperature_K, "ko", ms=3, label="Withheld digitized initial profile")
            ax[0].plot(g.x_m*1000, g.prediction_K, "-", label="Frozen network initial trace")
            ax[0].set(xlabel="Axial position (mm)", ylabel="Temperature (K)", title="Initial interior assessment")
            ax[0].legend(fontsize=8)
            ax[1].plot(metrics.time_s, metrics.RMSE_K, "o-", label="RMSE")
            ax[1].plot(metrics.time_s, metrics.bias_K, "s--", label="Signed bias")
            ax[1].set(xlabel="Withheld time (s)", ylabel="Error (K)", title="Same-run withheld interiors")
            ax[1].legend(fontsize=8)
            for axis in ax: axis.grid(alpha=.25)
            fig.savefig(run_dir / "brass_assessment_K.png", dpi=180)
            plt.show()

## 5. Replay and refine every completed nominal candidate

Each network's frozen initial trace is evolved with the independent conservative finite-difference/BDF solver, using the measured endpoint histories. Any initial endpoint correction is recorded. Terminal consistency is a fit to training observations; the other interior profiles are withheld same-run assessments.

The default selection includes **every completed nominal candidate**, not only the first one. Every reconstruction receives the same spatial grids and tighter time-integration check. Identical completed checks are cached. The proposed numerical floor is an empirical numerical-sensitivity estimate for that candidate; it is not measurement uncertainty or a certified error bound. It is never accepted automatically for classical parameter selection.

In [ ]:
REPLAY_RUN_DIRS = []              # Empty: all completed candidates in this exact nominal plan.
allowed = {p.resolve() for p in completed_nominal_runs}
replay_runs = [_project_path(p) for p in REPLAY_RUN_DIRS] if REPLAY_RUN_DIRS else list(completed_nominal_runs)
if any(p.resolve() not in allowed for p in replay_runs):
    raise ValueError("Replay selection must belong to the completed compact nominal plan.")
replay_summary = []
observed_numerical_floor = None    # Optional classical branch must load one reviewed floor explicitly.
if RUN_REPLAY or RUN_REFINEMENT:
    for run_dir in replay_runs:
        try:
            replay = brass_done.replay_cached(run_dir, observations, heldout,
                nx=REPLAY_MESHES[-1], rtol=REPLAY_RTOL, atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP)
            replay_summary.append({"run":run_dir.name, "cache":replay["cache_status"],
                "terminal_RMSE_K":replay["terminal_metrics"]["RMSE_K"],
                "maximum_initial_endpoint_correction_K":float(np.max(np.abs(replay["initial_boundary_correction_K"])))})
            print(run_dir.name, "independent replay:", replay["cache_status"])
            display(replay["heldout_metrics"])
        except Exception as exc:
            analysis_failures.append({"run":run_dir.name,"step":"independent_replay","error":str(exc)})
            print("Replay failed:",run_dir.name,str(exc))
    display(pd.DataFrame(replay_summary))
else:
    print("Independent replay disabled.")

In [ ]:
refinement_summary = []
if RUN_REFINEMENT:
    for run_dir in replay_runs:
        try:
            refinement, floor_record, cache_status = brass_done.refinement_cached(
                run_dir, observations, heldout, meshes=REPLAY_MESHES, rtol=REPLAY_RTOL,
                atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP, guard_factor=NUMERICAL_FLOOR_FACTOR)
            refinement_summary.append({"run":run_dir.name,"cache":cache_status,**floor_record})
            print(run_dir.name, "mesh/time refinement:", cache_status)
            display(refinement)
        except Exception as exc:
            analysis_failures.append({"run":run_dir.name,"step":"replay_refinement","error":str(exc)})
            print("Refinement failed:",run_dir.name,str(exc))
    display(pd.DataFrame(refinement_summary))
else:
    print("Mesh/time refinement disabled.")
write_json(BRASS_RESULTS / "brass_postprocessing_failures.json", analysis_failures)
if analysis_failures:
    # Preserve a cumulative failure history even if a later audit-only pass has no new errors.
    with (BRASS_RESULTS / "brass_postprocessing_failure_history.jsonl").open("a", encoding="utf-8") as log:
        for failure in analysis_failures:
            log.write(json.dumps(failure) + "\n")
    display(pd.DataFrame(analysis_failures))
    print("Retain these failures in the completion record; they are not silently discarded.")

## 6. Optional classical comparison

Both branches below require an explicit `UNCERTAINTY_K`, its provenance, and a reviewed numerical-floor study. Setting a value is an assumption unless independently documented. A per-point absolute bound \(b_K\) gives \(\|\epsilon\|_2\le\sqrt{48}\,b_K/190\); an RMS scenario gives the same scale numerically but is not a guaranteed bound.

- **Nonlinear Tikhonov:** fits a piecewise-linear initial profile to the actual interior terminal points. Its knot count, value bounds, regularization grid and forward resolution require sensitivity checks. Initial truth never enters parameter selection.
- **QR closure audit:** compares the archived-style mixed closure with a Navier closure. These are different regularizations. QR interpolates the sensors onto a full mesh; that extra preprocessing and its round-trip error are reported. A stable candidate with a small terminal discrepancy is not automatically a correct initial reconstruction. This one-dimensional audit does not validate the original two-/three-dimensional FreeFEM results.

The diagnostic safety/value intervals below are explicit numerical/regularization choices, not newly established physical bounds. Refine them and the parameter grids after inspecting stability and data fit. No classical run is silently promoted to a verified publication result.

In [ ]:
# Optional loading of a previously executed floor study after a notebook restart.
NUMERICAL_FLOOR_RECORD = None       # e.g. a run's brass_numerical_floor_study.json
if NUMERICAL_FLOOR_RECORD is not None:
    loaded_floor = read_json(_project_path(NUMERICAL_FLOOR_RECORD))
    if loaded_floor.get("case") != "brass" or loaded_floor.get("observation_sha256") != OBSERVATION_HASH:
        raise ValueError("The numerical-floor record does not belong to these brass observations.")
    observed_numerical_floor = loaded_floor.get("proposed_numerical_noise_floor")

classical_ready = False
classical_observations = None
if RUN_NONLINEAR_TIKHONOV or RUN_QR_AUDIT:
    missing = []
    if UNCERTAINTY_K is None: missing.append("documented positive UNCERTAINTY_K")
    if not UNCERTAINTY_PROVENANCE.strip(): missing.append("UNCERTAINTY_PROVENANCE")
    if observed_numerical_floor is None or not np.isfinite(observed_numerical_floor) or observed_numerical_floor <= 0:
        missing.append("positive floor from executed spatial/time replay checks")
    if not ACCEPT_OBSERVED_NUMERICAL_FLOOR: missing.append("reviewed numerical floor (ACCEPT_OBSERVED_NUMERICAL_FLOOR)")
    if missing:
        print("Classical fits not started; missing:", "; ".join(missing))
    else:
        classical_observations = baseline_observations(observations, UNCERTAINTY_K, UNCERTAINTY_INTERPRETATION)
        classical_ready = True
        write_json(BRASS_RESULTS / "classical_assumptions.json", dict(
            uncertainty_K=UNCERTAINTY_K, interpretation=UNCERTAINTY_INTERPRETATION,
            provenance=UNCERTAINTY_PROVENANCE, numerical_noise_floor=observed_numerical_floor,
            uncertainty_is_automatically_measured=False))
else:
    print("Classical fits disabled; unknown noise is not silently treated as zero.")

In [ ]:
# Truth-free nonlinear Tikhonov screen. Expand/refine after the first diagnostic.
TIKHONOV_KNOTS = (7, 11)
TIKHONOV_NX = 101
TIKHONOV_ALPHAS = np.logspace(-8, -1, 8)
TIKHONOV_MAX_NFEV = 60
TIKHONOV_VALUE_BOUNDS = (-0.2, 1.5)  # Explicit prior bounds in theta; vary them, do not call measured bounds.
if RUN_NONLINEAR_TIKHONOV and classical_ready:
    out = BRASS_RESULTS / "classical" / "nonlinear_tikhonov"
    out.mkdir(parents=True, exist_ok=True)
    for knots in TIKHONOV_KNOTS:
        result = nonlinear_tikhonov(problem, classical_observations, n_knots=knots,
            alphas=TIKHONOV_ALPHAS, nx=TIKHONOV_NX, numerical_noise_floor=observed_numerical_floor,
            rtol=REPLAY_RTOL, atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP,
            max_nfev=TIKHONOV_MAX_NFEV, value_bounds=TIKHONOV_VALUE_BOUNDS)
        candidate = out / f"knots{knots}_nx{TIKHONOV_NX}"
        candidate.mkdir(exist_ok=True)
        pd.DataFrame(result["path"]).to_csv(candidate / "parameter_path.csv", index=False)
        np.savez_compressed(candidate / "frozen_candidate.npz", x=result["x"], theta0=result["initial"],
                            terminal=result["terminal"], coefficients=result["coefficients"])
        write_json(candidate / "candidate_selection.json", dict(
            selected_parameter=result["selected_parameter"], selection_status=result["selection_status"],
            residual_norm=result["residual_norm"], target_residual=result["target_residual"],
            value_bounds=TIKHONOV_VALUE_BOUNDS, n_knots=knots, nx=TIKHONOV_NX,
            selection_uses_initial_truth=False, observation_sha256=OBSERVATION_HASH,
            evidence_status="data_selected_candidate_pending_solver_and_provenance_checks"))
        print("Knots:", knots, "status:", result["selection_status"])
        # Initial/intermediate assessment occurs only AFTER the data-selected candidate was saved.
        if not result["path"][result["selected_index"]]["optimizer_success"]:
            print("No converged selected optimizer candidate; keep the path as a failure diagnostic.")
            continue
        chosen_initial = lambda X, r=result: np.interp(X[:,0], r["x"], r["initial"])
        assessed = replay_initial(problem, chosen_initial, observations, heldout, nx=REPLAY_MESHES[-1],
            rtol=REPLAY_RTOL, atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP)
        assessed["heldout_metrics"].to_csv(candidate / "brass_classical_replay_metrics_by_time.csv", index=False)
        display(pd.DataFrame(result["path"]))

In [ ]:
QR_GAMMAS = np.unique(np.r_[np.logspace(-5, -1, 9), 5e-3]) # Includes archived manually chosen value.
QR_RESOLUTIONS = ((81, 0.005), (161, 0.0025))
QR_CLOSURES = ("archived", "navier")
if RUN_QR_AUDIT and classical_ready:
    out = BRASS_RESULTS / "classical" / "qr_closure_audit"
    out.mkdir(parents=True, exist_ok=True)
    summary_rows = []
    for closure in QR_CLOSURES:
        for nx, dt in QR_RESOLUTIONS:
            candidates = [solve_qr_1d(problem, classical_observations, gamma=float(gamma),
                nx=nx, dt=dt, closure=closure, safety_bounds=(-0.2, 1.5),
                forward_options=dict(rtol=REPLAY_RTOL, atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP))
                for gamma in QR_GAMMAS]
            interpolation_scale = max(r["sensor_to_mesh_backprojection_residual"] for r in candidates)
            choice = select_qr_by_discrepancy(candidates, classical_observations["noise_norm"],
                                             observed_numerical_floor + interpolation_scale)
            for r in candidates:
                summary_rows.append({k:r[k] for k in ("closure","gamma","nx_nodes","dt_actual","stable",
                    "failed_step","failure_reason","residual_norm","wall_seconds","sensor_to_mesh_backprojection_residual")})
            folder = out / f"{closure}_nx{nx}_dt{dt:g}"
            folder.mkdir(exist_ok=True)
            write_json(folder / "selection.json", {k:v for k,v in choice.items() if k != "selected_result"})
            selected = choice["selected_result"]
            print(closure, nx, dt, choice["selection_status"])
            if selected is not None and selected["stable"]:
                np.savez_compressed(folder / "frozen_candidate.npz", x=selected["x"], theta0=selected["theta0"],
                    mesh_terminal_input=selected["mesh_terminal_input"], terminal_replay=selected["terminal_forward"])
                write_json(folder / "candidate_protocol.json", dict(observation_sha256=OBSERVATION_HASH,
                    preprocessing=selected["preprocessing"], closure=closure,
                    selection_uses_initial_truth=False, evidence_status="diagnostic_candidate_not_verified_baseline"))
                selected_initial = lambda X, r=selected: np.interp(X[:,0], r["x"], r["theta0"])
                assessed = replay_initial(problem, selected_initial, observations, heldout, nx=REPLAY_MESHES[-1],
                    rtol=REPLAY_RTOL, atol=REPLAY_ATOL, max_step=REPLAY_MAX_STEP)
                assessed["heldout_metrics"].to_csv(folder / "brass_QR_replay_metrics_by_time.csv", index=False)
    qr_table = pd.DataFrame(summary_rows)
    qr_table.to_csv(out / "gamma_mesh_time_closure_audit.csv", index=False)
    display(qr_table)
else:
    print("QR gamma/closure audit not started.")

## 7. Optional inverse sensitivity runs — deferred

These six assumed scenarios are outside the nine-run nominal budget: diffusivity −10%, −5%, +5%, +10%, and common endpoint-history shifts −1 K and +1 K. They do not represent measured uncertainty or confidence intervals. The branch remains off by default.

If explicitly enabled later, `MAX_NEW_SENSITIVITY_JOBS` applies after completed jobs are removed. Each scenario uses the unchanged terminal vector and corresponding nominal method/seed/budget for comparison. All retained scenario candidates are frozen before withheld assessment.

In [ ]:
scenarios = sensitivity_scenarios(problem)
display(pd.DataFrame([{"scenario":s["scenario"], "assumption":s["assumption"], "Fo":s["problem"]["Fo"]} for s in scenarios]))
SENSITIVITY_PLAN = [job for job in retraining_plan(problem, training_seeds=SENSITIVITY_SEEDS,
    methods=SENSITIVITY_METHODS, include_sensitivity=True, profile=PROFILE)
    if job["scenario"] != "nominal" and (not SENSITIVITY_NAMES or job["scenario"] in SENSITIVITY_NAMES)]
print("Optional perturbation jobs:",len(SENSITIVITY_PLAN),"| Pending-job limit:",MAX_NEW_SENSITIVITY_JOBS)
if RUN_SENSITIVITY:
    sensitivity_inventory, sensitivity_outcomes = brass_done.train_missing(
        SENSITIVITY_PLAN, observations, BRASS_RESULTS / "sensitivity", max_new_jobs=MAX_NEW_SENSITIVITY_JOBS)
    display(sensitivity_outcomes)
    completed_sensitivity = []
    for row in sensitivity_inventory[sensitivity_inventory.status.eq("completed")].itertuples(index=False):
        job = SENSITIVITY_PLAN[int(row.job_index)]
        run_dir = Path(row.run_dir)
        write_json(run_dir / "sensitivity_assumption.json", dict(scenario=job["scenario"],
            assumption=job["assumption"], probability_model=None, is_measured_uncertainty=False))
        brass_done.freeze_candidate_portable(run_dir)
        completed_sensitivity.append(run_dir)
    for run_dir in completed_sensitivity:
        metrics, points, cache_status = brass_done.assessment_cached(run_dir, observations, heldout)
        print(run_dir.name, "assessment:", cache_status)
        display(metrics)
else:
    print("Sensitivity retraining deferred; the compact production plan includes nominal runs only.")

## Completion record for the paper

The article's brass section can be finalized only after the evidence below is reviewed. A completed notebook execution is not a substitute for the source-provenance checks.

| Item | Evidence to retain |
|---|---|
| Observation contract | Hashes, 48 terminal interiors, prescribed endpoint histories, withheld-role table |
| Nominal reconstructions | Production configurations, completed seeds, saved final weights and frozen candidate identities |
| Physical errors | `brass_metrics_by_time.csv`, plots in kelvin, bias and maximum error, normalization definition |
| Independent consistency | Replayed fields, terminal training-data discrepancy, independent withheld-time errors |
| Numerical accuracy | Spatial and BDF-tolerance trends; scope and basis of any accepted numerical floor |
| Classical comparison | Parameter paths, optimizer/stability failures, attained or unattained discrepancy, closure and interpolation effects |
| Sensitivity | Actual perturbed inverse runs; all assumptions stated; no invented probability distribution |
| Experimental provenance | Calibration dependence, original/digitized data status, uncertainty and schematic attribution |

**Reporting rule:** describe the intermediate profiles as *withheld interiors from the same experimental run*. If coefficient calibration uses those profiles or the initial field, disclose that dependence and present this case as an illustrative consistency study. Keep the manuscript placeholder until the remaining checks are actually resolved.